In [2]:
import pandas as pd

df = pd.read_csv('../data/filtered_complaints.csv')
df.columns = df.columns.str.strip()

# We sample n rows per group and combine them
n_per_group = int(15000 / 4)
sampled_chunks = []

for product in df['Product'].unique():
    group = df[df['Product'] == product]
    sampled_chunks.append(group.sample(n=min(len(group), n_per_group)))

# 3. Concatenate and reset index
sample_df = pd.concat(sampled_chunks).reset_index(drop=True)

# 4. Verify
print("Available columns now:", sample_df.columns.tolist())

# Now proceed with your loop, 'Product' will definitely be there

Available columns now: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count']


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # Balances context window vs. retrieval precision
    chunk_overlap=50,  # Ensures continuity between chunks
    length_function=len
)

# Create chunks
docs = []
for index, row in sample_df.iterrows():
    chunks = text_splitter.split_text(row['Consumer complaint narrative'])
    for c in chunks:
        docs.append({"text": c, "metadata": {"id": row['Complaint ID'], "product": row['Product']}})

In [4]:
print(sample_df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count']


Embedding and vector storage

In [4]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize local embeddings (it will now use your cached model automatically)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the Vector Store
vector_db = Chroma.from_texts(
    texts=[d['text'] for d in docs],
    metadatas=[d['metadata'] for d in docs],
    embedding=embeddings,
    persist_directory="vector_store/"
)

print("Vector store successfully indexed and saved to vector_store/ directory.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store successfully indexed and saved to vector_store/ directory.
